# Simple baselines + format checker + scorer

A quick walk-through of the **majority** and **random** baselines for both subtasks, followed by the **format checker** and the **scorer**. These are the fastest way to produce a valid submission file and to verify the end-to-end pipeline.

**Prerequisite:** `python data/download_data.py` has been run.

In [1]:
import sys, subprocess
from pathlib import Path

TASK1 = Path.cwd().resolve()
while TASK1.name != 'task1' and TASK1.parent != TASK1:
    TASK1 = TASK1.parent
DATA = TASK1 / 'data' / 'splits'
PRED = TASK1 / 'predictions'
PRED.mkdir(exist_ok=True)

def sh(*args):
    print('$', ' '.join(args))
    print(subprocess.check_output([sys.executable, *args], text=True, cwd=TASK1))

## 1. Majority baseline

Predicts the most-frequent class from the training set on every record.

* Subtask 1A: predicts `Not Hateful` (the larger class) for every meme.
* Subtask 1B: predicts the single most-frequent hateful sub-type (`Mocking` on the released data).


In [2]:
sh('baselines/majority_baseline.py', '--subtask', '1a',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'majority_1a.tsv'),
   '--run-id', 'majority')

sh('baselines/majority_baseline.py', '--subtask', '1b',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'majority_1b.jsonl'))


$ baselines/majority_baseline.py --subtask 1a --train /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/train.jsonl --target /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev_test.jsonl --out /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a.tsv --run-id majority


INFO loaded train=3500  target=500
INFO majority class = 'Not Hateful'
INFO loaded train=3500  target=500
INFO majority hateful subtype = 'Mocking'


Wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a.tsv

$ baselines/majority_baseline.py --subtask 1b --train /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/train.jsonl --target /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev_test.jsonl --out /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1b.jsonl
Wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1b.jsonl




INFO loaded train=3500  target=500
INFO majority non-hateful subtype = 'Sarcasm'


## 2. Random baseline

Bernoulli draw with training-set priors (1A) or independent multi-label draws with per-class priors (1B). Useful as a sanity-check ceiling for trivial systems.

In [3]:
sh('baselines/random_baseline.py', '--subtask', '1a',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'random_1a.tsv'),
   '--run-id', 'random', '--seed', '42')

sh('baselines/random_baseline.py', '--subtask', '1b',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'random_1b.jsonl'), '--seed', '42')


INFO loaded train=3500  target=500
INFO p(Hateful) = 0.3783


$ baselines/random_baseline.py --subtask 1a --train /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/train.jsonl --target /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev_test.jsonl --out /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1a.tsv --run-id random --seed 42
Wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1a.tsv

$ baselines/random_baseline.py --subtask 1b --train /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/train.jsonl --target /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev_test.jsonl --out /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1b.jsonl --seed 42


INFO loaded train=3500  target=500
INFO hateful priors = {'Contempt': 0.081, 'Dehumanization': 0.187, 'Exclusion': 0.008, 'Incitement': 0.242, 'Inferiority': 0.043, 'Mocking': 0.533, 'Other': 0.014, 'Slurs': 0.19}


Wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1b.jsonl




INFO loaded train=3500  target=500
INFO non-hateful priors = {'Humor': 0.397, 'Other': 0.175, 'Sarcasm': 0.429}


## 3. Format check

Always run the format checker before submitting.

In [4]:
for subtask, ext in [('1a', 'tsv'), ('1b', 'jsonl')]:
    for kind in ('majority', 'random'):
        sh('format_checker/format_checker.py',
           '--subtask', subtask,
           '--predictions', str(PRED / f'{kind}_{subtask}.{ext}'))

$ format_checker/format_checker.py --subtask 1a --predictions /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a.tsv


INFO [subtask_1a] OK: 500 predictions, run_id='majority', labels in ['Hateful', 'Not Hateful']


OK

$ format_checker/format_checker.py --subtask 1a --predictions /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1a.tsv


INFO [subtask_1a] OK: 500 predictions, run_id='random', labels in ['Hateful', 'Not Hateful']


OK

$ format_checker/format_checker.py --subtask 1b --predictions /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1b.jsonl


INFO [subtask_1b] OK: 500 predictions, label vocab size=13


OK

$ format_checker/format_checker.py --subtask 1b --predictions /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/random_1b.jsonl


INFO [subtask_1b] OK: 500 predictions, label vocab size=13


OK



OK

OK



## 4. Local scoring against `dev.jsonl`

The dev split is labelled, so we can compute metrics on it locally. (For `dev_test` the leaderboard is the only source of metrics during the development phase.)

We re-run the baselines with `--target dev.jsonl` to align the prediction IDs with the gold IDs.

In [ ]:
sh('baselines/majority_baseline.py', '--subtask', '1a',
   '--train', str(DATA / 'train.jsonl'), '--target', str(DATA / 'dev.jsonl'),
   '--out', str(PRED / 'majority_1a_dev.tsv'), '--run-id', 'majority_dev')

sh('scorer/scorer.py', '--subtask', '1a',
   '--gold', str(DATA / 'dev.jsonl'),
   '--predictions', str(PRED / 'majority_1a_dev.tsv'))

$ baselines/majority_baseline.py --subtask 1a --train /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/train.jsonl --target /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev.jsonl --out /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a_dev.tsv --run-id majority_dev


INFO loaded train=3500  target=500
INFO majority class = 'Not Hateful'


Wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a_dev.tsv

$ scorer/scorer.py --subtask 1a --gold /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/data/splits/dev.jsonl --predictions /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/task1/predictions/majority_1a_dev.tsv


Same for Subtask 1B:

In [ ]:
for subtask in ('1b',):
    sh('baselines/majority_baseline.py', '--subtask', subtask,
       '--train', str(DATA / 'train.jsonl'), '--target', str(DATA / 'dev.jsonl'),
       '--out', str(PRED / f'majority_{subtask}_dev.jsonl'))
    sh('scorer/scorer.py', '--subtask', subtask,
       '--gold', str(DATA / 'dev.jsonl'),
       '--predictions', str(PRED / f'majority_{subtask}_dev.jsonl'))


Your baselines should beat these majority numbers \u2014 if not, something is wrong with your training loop.